# 第7课：建立Unhedged Profit基准

本课使用第6课已经生成的10,000个产量与cash-price共同情景，计算代表性Iowa玉米农场在**完全不套保**时的收入与利润。

这条基准线非常重要：后面所有固定比例和July adaptive策略，都必须与同一批情景下的unhedged结果比较。

## 0. 这一课的四个公式

设农场面积为 $A$，最终单产为 $Y_i$，收获期现金价格为 $P_i$，每英亩生产成本为 $C$。

$$Production_i=Y_i\times A$$

$$CashRevenue_i=Production_i\times P_i$$

$$TotalProductionCost=C\times A$$

$$UnhedgedProfit_i=CashRevenue_i-TotalProductionCost$$

因为完全不套保，所以没有futures P&L、transaction cost或margin financing cost。

## 1. 农场与成本设定

- 代表性农场面积：1,000 acres，这是本项目的规模假设；
- 生产成本：$911.98/acre；
- 成本来源：Iowa State University Ag Decision Maker, **Estimated Costs of Crop Production in Iowa–2026, File A1-20**；
- 选择项目：Corn Following Soybeans，211 bu/acre预算列。

选择211 bu/acre列，是因为它最接近本项目2026 trend yield约210.25 bu/acre。

本模型把$911.98/acre视为固定成本。现实中部分收获、运输或投入成本可能随产量变化，这是需要披露的简化假设。

## 2. 导入工具并锁定参数

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

N_SIMULATIONS = 10_000
FARM_ACRES = 1_000
PRODUCTION_COST_PER_ACRE = 911.98
TOTAL_PRODUCTION_COST = FARM_ACRES * PRODUCTION_COST_PER_ACRE
LOWER_TAIL_PROBABILITY = 0.05
Z_95 = 1.96

print('模拟次数:', N_SIMULATIONS)
print('农场面积:', FARM_ACRES, 'acres')
print(f'生产成本: ${PRODUCTION_COST_PER_ACRE:,.2f}/acre')
print(f'农场总生产成本: ${TOTAL_PRODUCTION_COST:,.2f}')

## 3. 读取第6课共同情景

请先运行第6课，使当前工作目录中存在：

`lesson_06_outputs/cash_price_scenarios_10000.csv`

In [ ]:
candidate_paths = [
    Path.cwd() / 'lesson_06_outputs' / 'cash_price_scenarios_10000.csv',
    Path.cwd().parent / 'lesson_06_outputs' / 'cash_price_scenarios_10000.csv',
]

lesson6_path = next((p for p in candidate_paths if p.exists()), None)

if lesson6_path is None:
    raise FileNotFoundError(
        '没有找到第6课CSV。请先运行Lesson_06 Notebook的全部单元格，'
        '并保留lesson_06_outputs文件夹。'
    )

scenarios = pd.read_csv(lesson6_path)
print('读取文件:', lesson6_path)
print('行数:', len(scenarios))
print('列数:', len(scenarios.columns))

## 4. 检查输入列

本课真正进入收入公式的两个随机变量是：

- `final_yield_bu_per_acre`；
- `cash_price_usd_per_bushel`。

其他列继续保留，确保所有未来策略面对完全相同的情景。

In [ ]:
required_columns = [
    'scenario_id',
    'weather_source_year',
    'yield_residual_source_year',
    'price_residual_source_year',
    'july_pdsi',
    'july_yield_forecast_bu_per_acre',
    'final_yield_bu_per_acre',
    'preseason_futures_usd_per_bushel',
    'july_futures_usd_per_bushel',
    'harvest_futures_usd_per_bushel',
    'basis_usd_per_bushel',
    'cash_price_usd_per_bushel',
]

missing = [c for c in required_columns if c not in scenarios.columns]
assert not missing, f'缺少列: {missing}'
assert len(scenarios) == N_SIMULATIONS
assert scenarios['scenario_id'].is_unique
assert scenarios[required_columns].isna().sum().sum() == 0

print('第6课输入检查通过。')
print(scenarios[[
    'scenario_id', 'final_yield_bu_per_acre', 'cash_price_usd_per_bushel'
]].head(5).round(4).to_string(index=False))

## 5. 计算农场实际产量

第6课的yield单位是`bushels per acre`，乘以1,000 acres后才是全农场bushels。

$$ActualProduction_i=FinalYield_i\times1{,}000$$

In [ ]:
scenarios['actual_production_bushels'] = (
    scenarios['final_yield_bu_per_acre'] * FARM_ACRES
)

print(scenarios[[
    'scenario_id',
    'final_yield_bu_per_acre',
    'actual_production_bushels',
]].head(10).round(3).to_string(index=False))

## 6. 计算Cash Revenue

$$CashRevenue_i=ActualProduction_i\times CashPrice_i$$

这里的revenue只来自把实际收获的玉米卖给当地买家，不含任何期货损益。

In [ ]:
scenarios['cash_revenue_usd'] = (
    scenarios['actual_production_bushels']
    * scenarios['cash_price_usd_per_bushel']
)

scenarios['cash_revenue_usd_per_acre'] = (
    scenarios['cash_revenue_usd'] / FARM_ACRES
)

print(scenarios[[
    'scenario_id',
    'actual_production_bushels',
    'cash_price_usd_per_bushel',
    'cash_revenue_usd',
]].head(10).round(2).to_string(index=False))

## 7. 扣除Production Cost得到Unhedged Profit

$$UnhedgedProfit_i=CashRevenue_i-911.98\times1{,}000$$

每英亩版本可以直接写成：

$$UnhedgedProfitPerAcre_i=FinalYield_i\times CashPrice_i-911.98$$

In [ ]:
scenarios['production_cost_usd'] = TOTAL_PRODUCTION_COST
scenarios['production_cost_usd_per_acre'] = PRODUCTION_COST_PER_ACRE

scenarios['unhedged_profit_usd'] = (
    scenarios['cash_revenue_usd'] - scenarios['production_cost_usd']
)

scenarios['unhedged_profit_usd_per_acre'] = (
    scenarios['unhedged_profit_usd'] / FARM_ACRES
)

print(scenarios[[
    'scenario_id',
    'cash_revenue_usd',
    'production_cost_usd',
    'unhedged_profit_usd',
    'unhedged_profit_usd_per_acre',
]].head(10).round(2).to_string(index=False))

## 8. 用Scenario 1手算验证

In [ ]:
row1 = scenarios.iloc[0]

manual_production = row1['final_yield_bu_per_acre'] * FARM_ACRES
manual_revenue = manual_production * row1['cash_price_usd_per_bushel']
manual_profit = manual_revenue - TOTAL_PRODUCTION_COST

print(f"Final yield = {row1['final_yield_bu_per_acre']:.6f} bu/acre")
print(f'Production = {manual_production:,.3f} bushels')
print(f"Cash price = ${row1['cash_price_usd_per_bushel']:.6f}/bu")
print(f'Cash revenue = ${manual_revenue:,.2f}')
print(f'Total cost = ${TOTAL_PRODUCTION_COST:,.2f}')
print(f'Unhedged profit = ${manual_profit:,.2f}')

assert np.isclose(manual_production, row1['actual_production_bushels'])
assert np.isclose(manual_revenue, row1['cash_revenue_usd'])
assert np.isclose(manual_profit, row1['unhedged_profit_usd'])
print('Scenario 1手算验证通过。')

## 9. 必须通过的公式检查

In [ ]:
assert len(scenarios) == N_SIMULATIONS
assert scenarios['scenario_id'].is_unique
assert scenarios[[
    'actual_production_bushels',
    'cash_revenue_usd',
    'production_cost_usd',
    'unhedged_profit_usd',
    'unhedged_profit_usd_per_acre',
]].isna().sum().sum() == 0

assert np.allclose(
    scenarios['actual_production_bushels'],
    scenarios['final_yield_bu_per_acre'] * FARM_ACRES,
)
assert np.allclose(
    scenarios['cash_revenue_usd'],
    scenarios['actual_production_bushels'] * scenarios['cash_price_usd_per_bushel'],
)
assert np.allclose(scenarios['production_cost_usd'], TOTAL_PRODUCTION_COST)
assert np.allclose(
    scenarios['unhedged_profit_usd'],
    scenarios['cash_revenue_usd'] - TOTAL_PRODUCTION_COST,
)
assert np.allclose(
    scenarios['unhedged_profit_usd_per_acre'],
    scenarios['final_yield_bu_per_acre']
    * scenarios['cash_price_usd_per_bushel']
    - PRODUCTION_COST_PER_ACRE,
)

print('全部公式检查通过。')

## 10. 定义项目需要的风险指标

教授提醒我们：expected profit、variability和downside risk可能支持不同策略。因此从unhedged基准开始，我们统一计算：

- **Expected Profit**：10,000次利润的平均值；
- **Standard Deviation**：利润波动程度；
- **Probability of Loss**：利润低于0的比例；
- **5th Percentile**：只有5%的情景比它更差；
- **CVaR 5%**：最差5%情景的平均利润。

这里把profit的5th percentile直接报告为低尾阈值；数值越高，downside protection越好。

In [ ]:
def risk_summary(series):
    series = pd.Series(series)
    p5 = series.quantile(LOWER_TAIL_PROBABILITY)
    lower_tail = series[series <= p5]
    return pd.Series({
        'Mean': series.mean(),
        'Std Dev': series.std(ddof=1),
        'Probability Below Zero': (series < 0).mean(),
        'P5': p5,
        'CVaR 5%': lower_tail.mean(),
        'Median': series.median(),
        'P95': series.quantile(0.95),
        'Minimum': series.min(),
        'Maximum': series.max(),
    })

profit_summary = pd.DataFrame({
    'Profit: 1,000-acre farm ($)': risk_summary(scenarios['unhedged_profit_usd']),
    'Profit per acre ($/acre)': risk_summary(scenarios['unhedged_profit_usd_per_acre']),
}).T

print(profit_summary.round(4).to_string())

## 11. 计算Expected Profit的95% Monte Carlo Confidence Interval

10,000次模拟均值仍有Monte Carlo sampling error。使用：

$$CI_{95\%}=\bar{x}\pm1.96\frac{s}{\sqrt{n}}$$

这个区间衡量的是**模拟均值的数值精度**，不是单个农场未来利润会落入的范围。

In [ ]:
profit_per_acre = scenarios['unhedged_profit_usd_per_acre']
mean_profit_per_acre = profit_per_acre.mean()
standard_error = profit_per_acre.std(ddof=1) / np.sqrt(N_SIMULATIONS)
ci_low = mean_profit_per_acre - Z_95 * standard_error
ci_high = mean_profit_per_acre + Z_95 * standard_error

print(f'Expected profit = ${mean_profit_per_acre:.4f}/acre')
print(f'Monte Carlo standard error = ${standard_error:.4f}/acre')
print(f'95% CI = [${ci_low:.4f}, ${ci_high:.4f}] per acre')

## 12. 为什么高产不一定代表高利润？

模型中高产往往伴随较低的玉米价格，这叫天然对冲关系。我们查看yield、cash price和unhedged profit之间的相关系数。

In [ ]:
relationship_checks = pd.Series({
    'Corr(Final Yield, Cash Price)': scenarios['final_yield_bu_per_acre'].corr(
        scenarios['cash_price_usd_per_bushel']
    ),
    'Corr(Final Yield, Unhedged Profit)': scenarios['final_yield_bu_per_acre'].corr(
        scenarios['unhedged_profit_usd_per_acre']
    ),
    'Corr(Cash Price, Unhedged Profit)': scenarios['cash_price_usd_per_bushel'].corr(
        scenarios['unhedged_profit_usd_per_acre']
    ),
})

print(relationship_checks.round(6).to_string())

## 13. 画出Unhedged Profit分布

红线是break-even，也就是profit = 0。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(
    scenarios['unhedged_profit_usd_per_acre'],
    bins=40,
    color='#5B9BD5',
    edgecolor='white',
)
axes[0].axvline(0, color='#C00000', linestyle='--', label='Break-even')
axes[0].axvline(
    mean_profit_per_acre,
    color='black',
    linestyle=':',
    label='Mean',
)
axes[0].set_title('Unhedged Profit Distribution')
axes[0].set_xlabel('USD per acre')
axes[0].set_ylabel('Number of simulations')
axes[0].legend()

axes[1].scatter(
    scenarios['final_yield_bu_per_acre'],
    scenarios['cash_price_usd_per_bushel'],
    c=scenarios['unhedged_profit_usd_per_acre'],
    cmap='RdYlGn',
    s=9,
    alpha=0.45,
)
axes[1].set_title('Yield, Cash Price and Profit')
axes[1].set_xlabel('Final yield (bu/acre)')
axes[1].set_ylabel('Cash price ($/bu)')

plt.tight_layout()
plt.show()

## 14. 可重复性检查

In [ ]:
p5_profit = profit_per_acre.quantile(0.05)
cvar5_profit = profit_per_acre[profit_per_acre <= p5_profit].mean()

expected = {
    'mean_profit_per_acre': -1.5944606006325932,
    'std_profit_per_acre': 153.72927221656929,
    'probability_below_zero': 0.5543,
    'p5_profit_per_acre': -308.62718762014634,
    'cvar5_profit_per_acre': -338.7472297637651,
    'mean_cash_revenue_farm': 910385.5393993673,
}

actual = {
    'mean_profit_per_acre': profit_per_acre.mean(),
    'std_profit_per_acre': profit_per_acre.std(ddof=1),
    'probability_below_zero': (profit_per_acre < 0).mean(),
    'p5_profit_per_acre': p5_profit,
    'cvar5_profit_per_acre': cvar5_profit,
    'mean_cash_revenue_farm': scenarios['cash_revenue_usd'].mean(),
}

for key in expected:
    assert np.isclose(actual[key], expected[key], atol=1e-10), (key, actual[key], expected[key])

print('可重复性检查通过。')
print(pd.DataFrame({'Expected': expected, 'Actual': actual}).round(6).to_string())

## 15. 保存第7课结果

In [ ]:
OUTPUT_DIR = Path.cwd() / 'lesson_07_outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

scenario_path = OUTPUT_DIR / 'unhedged_baseline_10000.csv'
summary_path = OUTPUT_DIR / 'step_07_unhedged_summary.json'

scenarios.to_csv(scenario_path, index=False)

summary_for_json = {
    'strategy': 'Unhedged / Fixed 0%',
    'n_simulations': N_SIMULATIONS,
    'farm_acres': FARM_ACRES,
    'production_cost_usd_per_acre': PRODUCTION_COST_PER_ACRE,
    'total_production_cost_usd': TOTAL_PRODUCTION_COST,
    'cost_source': 'Iowa State Ag Decision Maker A1-20, 2026 Corn Following Soybeans, 211 bu/acre budget',
    'expected_cash_revenue_usd_1000_acre_farm': float(scenarios['cash_revenue_usd'].mean()),
    'profit_summary_usd_per_acre': {
        k: float(v) for k, v in risk_summary(profit_per_acre).items()
    },
    'mean_profit_95ci_usd_per_acre': [float(ci_low), float(ci_high)],
}

summary_path.write_text(json.dumps(summary_for_json, indent=2), encoding='utf-8')

print('已保存:', scenario_path)
print('已保存:', summary_path)

## 16. 本课结论

在当前数据和假设下，完全不套保的结果为：

- Expected profit约为 **−$1.59/acre**；
- Profit standard deviation约为 **$153.73/acre**；
- 亏损概率约为 **55.43%**；
- 5th-percentile profit约为 **−$308.63/acre**；
- 最差5%情景的平均利润约为 **−$338.75/acre**。

这不代表“不套保一定亏损”。它说明平均利润接近break-even，但利润分布很宽，极端低尾损失明显。

必须保留的限制：生产成本固定为$911.98/acre；未模拟政府补贴、作物保险、税费、质量折扣、储存决策或产量相关成本变化。

**下一课：加入第一个固定套保策略，并重点拆开教授要求的Initial Futures P&L与July Adjustment P&L。**